# Parrotlet-A 2.5 Pro — English Audio Transcription (Colab, fixed)

This is a cleaned-up version of your debugging notebook. It keeps everything that worked and fixes the one real bug that was producing wrong transcripts.

**Root cause of the wrong transcript:** the low-RAM decoder loading path uses `full_decoder.to_empty(device=device)`, which reallocates *all* parameters **and buffers** as uninitialized memory. Gemma3's embedding layer scales every embedding by `sqrt(hidden_size)`, stored as a *non-persistent* buffer (`embed_scale`) — so it's never in the checkpoint and never gets restored by `load_state_dict`. Every embedding (text **and** the projected audio) was being multiplied by garbage instead of `sqrt(hidden_size)`, corrupting everything downstream.

You'd actually already found this (see the `embed_scale` search you did) and tried to patch it — but the patch referenced `embed_layer`, a variable that was never defined anywhere, so it silently never applied (`NameError` if run on its own).

**What changed here:**
1. `fix_embed_scale()` is now a real, self-contained function that finds the actual embedding layer and resets its scale — applied automatically after loading, no matter which loading path is taken.
2. Tries the plain `from_pretrained` load first (its `__init__` sets `embed_scale` correctly on its own); only falls back to meta-device streaming if that OOMs — and the fallback path gets the fix applied explicitly.
3. Dropped the `model.encoder = model.encoder.float()` cell — it contradicted the bf16-only guidance and broke the dtype match `transcribe_fixed()` depends on.

*Note: I traced this from static code review — I don't have GPU/network access to re-run it against the real checkpoint here, so re-run start to finish and sanity-check the output.*

## 0. Disable TensorFlow backend (saves RAM before anything else loads)
`transformers` checks for a TensorFlow backend on import, which costs real RAM you don't need for a pure-PyTorch model. Run this first, before any other cell — including the pip installs.

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
print("TensorFlow backend disabled for transformers.")

## 1. Install / pin dependencies
Run once per fresh Colab VM. If prompted to restart the runtime after this, do so (Runtime → Restart runtime), then continue from cell 2.

In [ ]:
!pip install "transformers==4.52.0" --quiet
!pip install librosa soxr --quiet
!pip install -U accelerate bitsandbytes --quiet

import transformers
print("transformers version:", transformers.__version__)

## 2. Load the model
Tries a plain bf16 load first (correct `embed_scale` out of the box). If that OOMs on CPU RAM, falls back to meta-device streaming + 4-bit decoder — and explicitly re-fixes `embed_scale`, since that path is exactly what corrupted it before.

Do **not** call `.float()` on the encoder afterwards — keep it in bf16, `transcribe_fixed()` below is written to match whatever dtype the encoder is actually in.

In [ ]:
import os, glob, json, gc, sys, torch
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from transformers import (
    AutoModel, AutoTokenizer, AutoProcessor, AutoConfig,
    Gemma3ForConditionalGeneration,
)
from accelerate import init_empty_weights
from transformers.dynamic_module_utils import get_class_from_dynamic_module

MODEL_ID = "ekacare/parrotlet-a-2.5-pro"
device = "cuda" if torch.cuda.is_available() else "cpu"


def fix_embed_scale(full_decoder):
    """
    Gemma3's word-embedding layer scales every embedding by sqrt(hidden_size).
    That scale is a *non-persistent* buffer, so it's never in the checkpoint's
    safetensors files. If the decoder is ever materialized with to_empty()
    (meta-device loading) instead of a normal from_pretrained, that buffer
    comes back as uninitialized garbage instead of sqrt(hidden_size) -- which
    silently corrupts every embedding the decoder sees (text AND projected
    audio) and produces fluent-looking but completely wrong transcripts.
    This resets it to the correct value no matter which loading path ran.
    """
    hidden_size = full_decoder.config.text_config.hidden_size
    correct_scale = hidden_size ** 0.5
    embed_layer = full_decoder.model.language_model.embed_tokens

    current = getattr(embed_layer, "embed_scale", None)
    was_wrong = (
        current is None
        or not torch.is_tensor(current)
        or not torch.isclose(current.float().cpu(), torch.tensor(float(correct_scale)), rtol=1e-3)
    )
    embed_layer.embed_scale = torch.tensor(
        correct_scale, dtype=embed_layer.weight.dtype, device=embed_layer.weight.device
    )
    tag = "was corrupted -> fixed" if was_wrong else "already correct"
    print(f"embed_scale = {correct_scale:.4f}  ({tag})")
    return full_decoder


!free -h
print("Downloading repo snapshot...")
local_dir = snapshot_download(repo_id=MODEL_ID)
encoder_dir = os.path.join(local_dir, "encoder")
decoder_dir = os.path.join(local_dir, "decoder")
projector_dir = os.path.join(local_dir, "projector")

# ---- Encoder ----
print("Loading encoder...")
encoder = AutoModel.from_pretrained(
    encoder_dir, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True
).encoder.to(device)

combined_state_dict = {}
for fp in sorted(glob.glob(f"{encoder_dir}/*.safetensors")):
    combined_state_dict.update(load_file(fp))
encoder.load_state_dict(combined_state_dict)
encoder.eval()
del combined_state_dict
gc.collect(); torch.cuda.empty_cache()
print("Encoder loaded ->", device)

# ---- Decoder ----
decoder_config = AutoConfig.from_pretrained(decoder_dir)

try:
    print("Loading decoder (plain bf16)...")
    full_decoder = Gemma3ForConditionalGeneration.from_pretrained(
        decoder_dir, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True
    ).to(device)
    print("Decoder loaded via plain from_pretrained (embed_scale set correctly by __init__).")
except torch.cuda.OutOfMemoryError:
    print("Plain load OOM'd -- falling back to meta-device streaming...")
    gc.collect(); torch.cuda.empty_cache()

    with init_empty_weights():
        full_decoder = Gemma3ForConditionalGeneration(decoder_config)

    state_dict = {}
    for fp in glob.glob(f"{decoder_dir}/*.safetensors"):
        state_dict.update(load_file(fp))

    remapped = {}
    for k, v in state_dict.items():
        if k.startswith("language_model.model."):
            new_k = "model.language_model." + k[len("language_model.model."):]
        elif k.startswith("vision_tower."):
            new_k = "model." + k
        elif k.startswith("multi_modal_projector."):
            new_k = "model." + k
        else:
            new_k = k
        remapped[new_k] = v
    del state_dict
    gc.collect()

    embed_key = "model.language_model.embed_tokens.weight"
    if "lm_head.weight" not in remapped and embed_key in remapped:
        remapped["lm_head.weight"] = remapped[embed_key]

    full_decoder = full_decoder.to_empty(device=device)
    missing, unexpected = full_decoder.load_state_dict(remapped, strict=False)
    print(f"missing: {len(missing)}  |  unexpected: {len(unexpected)}")
    del remapped
    gc.collect(); torch.cuda.empty_cache()

    full_decoder = full_decoder.to(dtype=torch.bfloat16)

# Always verify/fix embed_scale, regardless of which path just ran --
# cheap, and this is the exact thing that silently broke last time.
fix_embed_scale(full_decoder)

del full_decoder.model.vision_tower
del full_decoder.model.multi_modal_projector
gc.collect(); torch.cuda.empty_cache()

decoder = full_decoder.language_model
decoder.eval()

tokenizer = AutoTokenizer.from_pretrained(decoder_dir)
processor = AutoProcessor.from_pretrained(encoder_dir)

# ---- Projector ----
print("Loading projector...")
with open(os.path.join(projector_dir, "config.json"), "r") as f:
    proj_cfg = json.load(f)

custom_module_name = [m for m in sys.modules if "modelling_speech" in m or "modeling_speech" in m]
if custom_module_name:
    speech_mod = sys.modules[custom_module_name[0]]
else:
    SpeechLLMClass = get_class_from_dynamic_module(
        class_reference=f"{MODEL_ID}--modelling_speech-llm.SpeechLLM",
        pretrained_model_name_or_path=MODEL_ID,
    )
    speech_mod = sys.modules[SpeechLLMClass.__module__]

SpeechLLM = speech_mod.SpeechLLM
load_projector = speech_mod.load_projector

encoder_dim = proj_cfg["encoder_dim"]
llm_dim = proj_cfg["llm_dim"]
linear_hidden_dim = proj_cfg["linear_hidden_dim"]
k = proj_cfg["encoder_projector_ds_rate"]
projector = load_projector(projector_dir, encoder_dim, llm_dim, linear_hidden_dim, k).to(device=device, dtype=torch.bfloat16)
projector.eval()

# ---- Assemble ----
sampling_rate = getattr(processor.feature_extractor, "sampling_rate", 16000)
model = SpeechLLM(decoder_config, encoder, projector, full_decoder, tokenizer, processor, sampling_rate)

print("=" * 70)
print("PARROTLET-A LOADED")
print("=" * 70)
print("Model class:", type(model))
print("encoder dtype:", next(model.encoder.parameters()).dtype)
print("decoder dtype:", next(model.decoder.parameters()).dtype)

## 3. Patched transcription function
Same logic as the model's own `transcribe()`, with the fix: casts the audio features to the encoder's actual dtype (bf16) before the forward pass, instead of upcasting the whole model to fp32. This only works correctly now that the encoder is *not* force-floated in cell 2 and `embed_scale` is guaranteed correct.

In [ ]:
import torch

def transcribe_fixed(model, audio, orig_sr, max_new_tokens=256, repetition_penalty=1.2, **gen_kwargs):
    device = next(model.parameters()).device
    model_dtype = next(model.encoder.parameters()).dtype  # bf16

    prompt = model.get_prompt()
    input_ids = torch.tensor(model.tokenizer(prompt, add_special_tokens=False)['input_ids'])
    input_attention_mask = torch.ones_like(input_ids)

    audio_token = model.tokenizer.convert_tokens_to_ids(model.audio_token)
    audio_pos = input_ids.tolist().index(audio_token)

    input_ids = input_ids.unsqueeze(0).to(device)
    input_attention_mask = input_attention_mask.unsqueeze(0).to(device)

    processed_audio = model.preprocess_audio(audio, orig_sr)

    audio_features = model.processor.feature_extractor(
        [processed_audio], sampling_rate=model.sampling_rate, return_tensors="pt"
    ).input_features
    audio_features = audio_features.to(device=device, dtype=model_dtype)  # the fix

    with torch.no_grad():
        audio_embeddings = model.encoder(audio_features).last_hidden_state
        projected_audio_embeddings = model.projector(audio_embeddings)

    input_embeddings = model.decoder.get_input_embeddings()(input_ids)
    batch_size, input_seq_len, embed_dim = input_embeddings.shape
    audio_seq_len = projected_audio_embeddings.shape[1]

    max_combined_len = input_seq_len + audio_seq_len - 1
    combined_embeddings = torch.zeros(batch_size, max_combined_len, embed_dim, device=device, dtype=input_embeddings.dtype)
    combined_attention_mask = torch.zeros(batch_size, max_combined_len, device=device, dtype=input_attention_mask.dtype)

    combined_embeddings[:, :audio_pos] = input_embeddings[:, :audio_pos]
    combined_attention_mask[:, :audio_pos] = input_attention_mask[:, :audio_pos]
    combined_embeddings[:, audio_pos:audio_pos+audio_seq_len] = projected_audio_embeddings
    combined_attention_mask[:, audio_pos:audio_pos+audio_seq_len] = 1

    suffix_start = audio_pos + 1
    suffix_len = input_seq_len - suffix_start
    out_start = audio_pos + audio_seq_len
    combined_embeddings[:, out_start:out_start+suffix_len] = input_embeddings[:, suffix_start:]
    combined_attention_mask[:, out_start:out_start+suffix_len] = input_attention_mask[:, suffix_start:]

    default_gen_kwargs = {
        'max_new_tokens': max_new_tokens,
        'do_sample': False,
        'repetition_penalty': repetition_penalty,
        'pad_token_id': model.tokenizer.pad_token_id,
        'eos_token_id': model.tokenizer.eos_token_id,
    }
    default_gen_kwargs.update(gen_kwargs)

    with torch.no_grad():
        outputs = model.decoder.generate(
            inputs_embeds=combined_embeddings,
            attention_mask=combined_attention_mask,
            **default_gen_kwargs
        )

    return model.tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print("transcribe_fixed() ready.")

## 4. Upload audio and transcribe
Upload an English recording (wav/mp3/etc). It's resampled to the model's required 16kHz before transcription.

In [ ]:
import librosa
from google.colab import files

uploaded = files.upload()
file_path = list(uploaded.keys())[0]

target_sr = model.sampling_rate  # 16000
audio_array, sr = librosa.load(file_path, sr=target_sr)

print(f"Loaded '{file_path}' -- {len(audio_array)/sr:.1f}s at {sr}Hz")

transcript = transcribe_fixed(model, audio_array, sr)

print("=" * 70)
print("TRANSCRIPT")
print("=" * 70)
print(transcript)

## 5. (Optional) Save transcript to a text file

In [ ]:
out_name = file_path.rsplit(".", 1)[0] + "_transcript.txt"
with open(out_name, "w", encoding="utf-8") as f:
    f.write(transcript)

print(f"Saved to {out_name}")

from google.colab import files as colab_files
colab_files.download(out_name)